# Simulating Calendar Aging Simulation from Literature
Fly, A., Wimarshana, B., Bin-Mat-Arishad, I., & Broad, R. J. (2025). Influence of periodic temperature variations on calendar ageing of lithium-ion batteries. Journal of Power Sources, 657, 238147.  
[https://doi.org/10.1016/j.jpowsour.2025.238147](https://doi.org/10.1016/j.jpowsour.2025.238147)  
[Data link](https://repository.lboro.ac.uk/articles/dataset/Underlying_data_Influence_of_periodic_temperature_variations_on_calendar_ageing_of_lithium-ion_batteries_/26870248?file=57211955)

In [ ]:
import pybamm
import numpy as np
import pickle
import pandas as pd
from scipy.integrate import cumulative_trapezoid
import matplotlib.pyplot as plt
import scipy.io as sio

# pybamm.set_logging_level("INFO")
pybamm.set_logging_level("WARNING")

data_DIR = "../data/"
exp_DIR = data_DIR + "lboro/"
out_DIR = data_DIR + "output/"
fig_DIR = "../figs/"
# %matplotlib widget

## Load Data

In [ ]:
file_name = exp_DIR + "lboro_rpt_days.mat"
mat_contents = sio.loadmat(file_name)
# print(mat_contents.keys())
lboro_rpt_days = mat_contents['lboro_rpt_days'][0]
file_name = exp_DIR + "lboro_rpt_cap.mat"
mat_contents = sio.loadmat(file_name)
# print(mat_contents.keys())
lboro_rpt_cap = mat_contents['Capacity_matrix']

## Cell Info
- Cell001: 100% SOC at $50^\circ C$ 
- Cell002: 100% SOC at $50^\circ C$ 
- Cell003: 50% SOC at $50^\circ C$ 
- Cell004: 50% SOC at $50^\circ C$ 
- Cell025: 100% SOC at $25^\circ C$ 
- Cell026: 100% SOC at $25^\circ C$ 
- Cell027: 50% SOC at $25^\circ C$ 
- Cell028: 50% SOC at $25^\circ C$ 

## Room Temperature

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(5,4))
ax.plot(lboro_rpt_days,lboro_rpt_cap[25-1,:])
ax.plot(lboro_rpt_days,lboro_rpt_cap[26-1,:])
ax.set_title("Calendar Aging @ Room Temp 100% SOC")
ax.set_xlabel("Days")
ax.set_ylabel("Capacity")

### 100% SOC Simulation

In [ ]:
# load existing LGM50 data set
parameter_values = pybamm.ParameterValues("Chen2020")
# model
model = pybamm.lithium_ion.DFN(
    {
        "SEI": "ec reaction limited",
    }
)

("Rest for 24 hours",)
days_max = lboro_rpt_days[-1]
experiment = pybamm.Experiment(
    [
        "Rest for 24 hours",
    ]*days_max,
)

In [ ]:
parameter_values.search("SEI")

In [ ]:
Temp = 25
parameter_values.update(
    {
        "Initial temperature [K]": 273.15+Temp,
        "Ambient temperature [K]": 273.15+Temp,
        "SEI kinetic rate constant [m.s-1]":  5e-8, #1.08494281e-16 , 
        "EC diffusivity [m2.s-1]":  1e-22,#8.30909086e-19,
        "SEI growth activation energy [J.mol-1]": 0,#1.58777981e+04,
        "Initial SEI thickness [m]": 5e-09,
        # "SEI resistivity [Ohm.m]": 30000.0,
        # "Negative electrode partial molar volume [m3.mol-1]": 7e-06,
    },
    check_already_exists=False,
)

In [ ]:
sim = pybamm.Simulation(
    model,
    experiment=experiment,
    parameter_values=parameter_values,
)
solution = sim.solve(initial_soc=1,save_at_cycles=50)

In [ ]:
# sorted(solution.summary_variables.all_variables)
# sum_vars = solution.summary_variables.get_summary_variables()
# pybamm.plot_summary_variables(solution)
day_sim = solution.summary_variables.cycle_number
cap_sim = solution.summary_variables["Capacity [A.h]"]

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(5,4))
ax.plot(lboro_rpt_days,lboro_rpt_cap[23-1,0]-lboro_rpt_cap[23-1,:],'k',marker="o")
ax.plot(lboro_rpt_days,lboro_rpt_cap[24-1,0]-lboro_rpt_cap[24-1,:],'k',marker="v")
ax.plot(day_sim,cap_sim[0]-cap_sim,'r--',linewidth=3)
ax.set_title("Calendar Aging @ Room Temp 100% SOC")
ax.set_xlabel("Days")
ax.set_ylabel("Capacity Loss [Ah]")
ax.legend(["Data 1","Data 2","Sim"])
fig.savefig(fig_DIR + "lboro_comp_1.png",dpi=600)

In [ ]:
dfsdfs

### 50% SOC Simulation

In [ ]:
sim = pybamm.Simulation(
    model,
    experiment=experiment,
    parameter_values=parameter_values,
)
solution1 = sim.solve(initial_soc=0.5,save_at_cycles=25)
day_sim1 = solution1.summary_variables.cycle_number
cap_sim1 = solution1.summary_variables["Capacity [A.h]"]

In [ ]:
np.size(lboro_rpt_cap)

In [ ]:
fig, ax = plt.subplots(1,1,figsize=(5,4))
ax.plot(lboro_rpt_days,lboro_rpt_cap[25-1,0]-lboro_rpt_cap[25-1,:],'k',marker="o")
ax.plot(lboro_rpt_days,lboro_rpt_cap[26-1,0]-lboro_rpt_cap[26-1,:],'k',marker="v")
ax.plot(day_sim1,cap_sim1[0]-cap_sim1,'r--',linewidth=3)
ax.set_title("Calendar Aging @ Room Temp 50% SOC")
ax.set_xlabel("Days")
ax.set_ylabel("Capacity Loss [Ah]")
ax.legend(["Data 1","Data 2","Sim"])
fig.savefig(fig_DIR + "lboro_comp_2.png",dpi=600)